# Basic RAG with LlamaIndex
Query a local text file using semantic node splitting and a vector index.

In [1]:
import os
from pathlib import Path

from get_api_key import get_api_key
from llama_index.core import Settings, VectorStoreIndex
from llama_index.core.node_parser import SemanticSplitterNodeParser
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI
from llama_index.readers.file import FlatReader

os.environ["OPENAI_API_KEY"] = get_api_key()

In [2]:
# Configure LlamaIndex settings
Settings.llm = OpenAI(model="gpt-3.5-turbo", max_tokens=100)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-ada-002")

In [3]:
# Semantic node splitter
splitter = SemanticSplitterNodeParser(
    buffer_size=1,
    breakpoint_percentile_threshold=95,
    embed_model=Settings.embed_model,
)

In [4]:
def parse_nodes(text_doc='sample_text.txt'):
    documents = FlatReader().load_data(Path(text_doc))
    return splitter.get_nodes_from_documents(documents)

In [5]:
# Vector index and query engine
nodes = parse_nodes()
index = VectorStoreIndex(nodes)
query_engine = index.as_query_engine(similarity_top_k=2)

2026-10-11 13:14:27,249 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-10-11 13:14:28,428 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


In [6]:
def query_doc(user_query):
    if not user_query or not user_query.strip():
        return None
    return query_engine.query(user_query)

In [7]:
response = query_doc("What computer did the author buy around 1980?")
print("Query Response:")
print(response)

2026-10-11 13:14:37,070 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-10-11 13:14:40,050 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Query Response:
The author purchased an Apple II computer around 1980.
